# Getting Started — 10-Minute Quickstart

## Why This Notebook Still Exists

This notebook remains the **fastest entry point** to FHS. It answers one concrete question in ~10 minutes:
**"Should we invest in this feature?"**

Use this notebook when you want to:
- define one feature,
- run one simulation,
- read Expected Value, Business Value Floor, P95,
- make a first go/no-go decision.

Use [T01 Distribution Guide](tutorial/01-distribution-guide.ipynb) when you want to:
- choose the right distribution (Normal, Lognormal, Beta),
- understand clipping bias and uncertainty effects,
- compare Business Value Floor / CVaR shifts across models.

## The Problem You're Solving

You've built a product feature. Now the question: **Should we invest the development cost?**

Most teams answer with a guess or a spreadsheet. This notebook does something different:
- **Input:** 3 realistic assumptions about how users will behave
- **Simulation:** 10,000 scenarios with Monte Carlo
- **Output:** A risk profile + conservative floor (not just an average)

> **Why 10,000 scenarios?** A single forecast is almost always wrong. A distribution of outcomes tells you what to *really* expect and what to budget for.

---

### How it works

| Step | What you do | Result |
|:--:|---|---|
| **1** | Define feature: expected users, conversion, uncertainty | Simulation input |
| **2** | Simulate 10,000 realistic scenarios | Distribution of outcomes |
| **3** | Read dashboard: expected value + business value floor | Decision support |

**Continue learning:** [T01 Distribution Guide](tutorial/01-distribution-guide.ipynb) → [02 Blockchain Case Study](02-blockchain-case-study.ipynb)


We roll the dice 10,000 times with random user numbers and observe which outcomes are common and which are rare.


The business value floor is the planning floor you can rely on; CVaR is the average of outcomes below that floor. In finance this floor is often called VaR 95% — here we lead with the product decision view.


📌 **Key Takeaway** — Most teams forecast a single number. You'll forecast a *distribution*: expected value, business value floor, and upside. This is how real risk decisions get made.


In [ ]:
from fhs import Feature
from fhs.application import BlockchainCaseStudyService
from fhs.notebook import notebook_setup
from fhs.presentation.notebook import (
    COLORS,
    plot_monte_carlo_comparison,
    plot_risk_profile,
    plot_uncertainty_comparison,
    show,
)

setup = notebook_setup("blockchain", show_status=False)
scenario = setup.scenario

if scenario is None:
    raise RuntimeError("Scenario setup could not be initialized")

show.info("✅ <b>Environment ready</b>")

---

## 1. Define Your Feature: 3 Simple Inputs

The simulation needs just three realistic assumptions about user behavior:

### Expected Users
**What it is:** How many users will encounter this feature?
- **Where it comes from:** Traffic projections, segment analysis, or product roadmap targets
- **Example:** Your app has 100K monthly active users → expect 10K in the target segment

### Conversion Rate
**What it is:** What fraction of users will complete the desired action (sign up, upgrade, purchase)?
- **Where it comes from:** Benchmarks, A/B tests from similar features, or industry reports
- **Example:** Your freemium app sees ~15% upgrade rate on average
- **Keep it realistic:** Use past data, not optimistic projections

### Uncertainty
**What it is:** How confident are you in these assumptions? (range: 0.1–0.5)
- **Where it comes from:** How much historical variance did you see? Market volatility? Technical risk?
- **Example:** 
  - **0.15 (Low):** You have strong data or similar past features (regulated market, mature product)
  - **0.30 (Medium):** Typical scenario with some unknowns (new user segment, market shift)
  - **0.50 (High):** Major unknowns (brand new feature, beta user base, unstable market)

> **What uncertainty means technically:** Uncertainty = 0.3 means the actual value stays within ±30% of your estimate in 99.7% of all scenarios. This defines a 3-sigma confidence band:
>
> `uncertainty = 0.3  →  std dev ≈ 10% of mean  →  99.7% band: [mean − 30%, mean + 30%]`
>
> You do not need to calculate this yourself — the simulation handles it. Just pick a level (Low / Medium / High) that matches your confidence in the inputs.

> **Note on distributions:** By default, we use a **Normal distribution** (bell curve). This works for most product features. For heavy-tailed scenarios (e.g., "most users do nothing, but some spend a lot"), upgrade to [Lognormal in T01](tutorial/01-distribution-guide.ipynb).

### How to adjust the feature input
1. Change `name`, `expected_users`, `conversion_rate`, and `uncertainty` in the next code cell.
2. Keep `conversion_rate` between `0` and `1`, `uncertainty` between `0.1` and `0.5`.
3. Execute the cell, then continue with the simulation below.


In [ ]:
checkout_feature = Feature(
    name="Optimierter Checkout-Flow",
    expected_users=10_000,
    conversion_rate=0.15,
    uncertainty=0.3,
    business_value_per_conversion=41.6,
    development_cost=75_000,
    installment_years=1,
)

show.info(f"✅ <b>Feature loaded:</b> {checkout_feature.name}")

In [ ]:
# ── Run simulation scenarios via application service ───────────
case = BlockchainCaseStudyService(seed=scenario.seed, scenarios=scenario.scenarios)
demo_year1 = case.simulate_year1({"D1": checkout_feature})
result = demo_year1["D1"].result

---

## Why Monte Carlo? — One Number Hides the Risk

Most business cases use a **single estimate**: "We expect 1,500 conversions worth €62,400."

That number is not wrong — but it hides the range of possible outcomes. What if the real result is 800 conversions? Or 2,200? A single number cannot tell you **how likely** a bad outcome is.

**Monte Carlo simulation** replaces the single guess with 10,000 realistic scenarios. Instead of one answer, you get a distribution — and you can see exactly how often the outcome falls below your break-even point.

> **Think of it this way:** A weather forecast that says "22°C tomorrow" is less useful than one that says "22°C expected, but a 30% chance it drops below 15°C." The simulation gives you the second kind of forecast — for your business decisions.

In [ ]:
# ── Why Monte Carlo? — Point Estimate vs. Distribution ─────────────
plot_monte_carlo_comparison(
    result,
    development_cost=checkout_feature.development_cost,
    business_value_per_conversion=checkout_feature.business_value_per_conversion,
);

In [ ]:
# ── Executive Summary Card ─────────────────────────────────────────
expected = result.expected_value
var95 = result.var_95
best = result.p95

risk_label, risk_verdict = result.risk_verdict()
_, light = result.risk_classification()

show.executive(
    f"📊 Executive Risk Summary: {checkout_feature.name}",
    [
        (f"{expected:,.0f}", "Expected Conversions", COLORS.secondary),
        (f"{var95:,.0f}", "Business Value Floor", COLORS.danger),
        (f"{best:,.0f}", "Best Case (P95)", COLORS.primary),
        (f"{result.std:,.0f}", "Spread (Std Dev)", COLORS.accent),
    ],
    footer=f"{light} <b>{risk_label} — {risk_verdict}</b>",
)

---

## 2. Read Your Risk Dashboard

The chart shows the distribution from 10,000 simulated results.
- **Expected Value**: Planning value (average).
- **Business Value Floor**: Conservative floor (95 out of 100 scenarios are above this).
- **P95**: Optimistic range.


In [ ]:
# ── Visual Risk Profile ─────────────────────────────────────────────
plot_risk_profile(result);

**How to read this dashboard**

**Left panel (Histogram):** The distribution of all 10,000 simulated outcomes. A narrow histogram = high confidence; a wide histogram = high uncertainty.

**Red dashed line:** Business Value Floor — your conservative minimum. Only 5% of scenarios fall below this.

**Right panel (KPI Card):** The executive summary. Use these three numbers:
- **Expected:** Plan your roadmap with this
- **Business Value Floor:** Budget with this
- **P95:** Share this as best-case upside

> **Decision rule:** If the business value floor is above your development cost, the feature is worth building. If it's close, review your uncertainty assumptions.

> **Learn more:** [T01 Distribution Guide](tutorial/01-distribution-guide.ipynb) explains how the business value floor shifts across distributions.


### Which metric should I use?

| I want to know... | Metric | Plain English |
|---|---|---|
| What do I plan for on average? | **Expected Value** | Basis for budget and roadmap |
| What is my conservative floor? | **Business Value Floor** | 95 out of 100 scenarios are above this |
| How bad can it get in the worst case? | **CVaR 95%** | Average of the worst 5% of scenarios |
| What are my delivery cost risks? | **CaR 95%** | Delivery cost in the 95th percentile (see [NB 06](06-blockchain-case-study-development-risk.ipynb)) |
| What do I lose if I cancel? | **Sunk Cost** | Money already spent, not recoverable (see [NB 06](06-blockchain-case-study-development-risk.ipynb)) |

> **Rule of thumb:** Use Expected Value for your roadmap, the business value floor for your budget, and CVaR 95% when presenting downside risk to the board.

---

## 3. Interpret the Numbers

### Your metrics at a glance

**Expected Value** — Your planning estimate (the average)
- **Use:** Base your roadmap timeline and budget on this
- **Why:** Represents the mean of all 10,000 scenarios

**Business Value Floor** — Your conservative floor (95% confidence)
- **Use:** This is what you should *budget for* to be safe
- **Why:** In 95 out of 100 simulations, you'll exceed this. Only 5% fall below.

**P95** — Your optimistic case (95th percentile)
- **Use:** Share this in board presentations as the upside
- **Why:** A realistic best-case scenario, not a unicorn outcome

**Std Dev** — Your volatility (spread of outcomes)
- **Use:** Assess how predictable this feature is
- **Why:** High spread = high uncertainty; low spread = high confidence

### How to present your findings

**For Finance (Budget Discussion):**
> "We expect **{Expected}** conversions with a conservative floor of **{Business Value Floor}**. We simulated 10,000 scenarios with realistic uncertainty assumptions."

**For the Executive Team (Risk Perspective):**
> "Best case: **{P95}** conversions. Most likely: **{Expected}**. Conservative floor: **{Business Value Floor}**. Our confidence level is **{Risk verdict from traffic light}**."

**For Product Management (Roadmap Planning):**
> "If uncertainty changes (market shift, technical pivot), this metric will shift proportionally. See [Uncertainty Comparison](01-getting-started.ipynb) above for sensitivity analysis."


---

## 4. When the Floor Isn't Enough: CVaR (The Worst-Case Average)

The business value floor tells you the **minimum you'll achieve in 95% of scenarios**. But what about the remaining 5%?

### The Business Question
When you present the business value floor to Finance, they ask: *"Okay, but if things go wrong—really wrong—how bad is it?"*

That's where **CVaR (Conditional Value at Risk)** comes in. It answers: *"In the worst 5% of scenarios, what's the average outcome?"*

| Metric | What it answers | Business use |
|--------|----------------|---------------|
| **Business Value Floor** | "What's the floor for most scenarios?" | **Budgeting:** Plan conservatively |
| **CVaR 95%** | "What's the *average* in the tail risk?" | **Contingency planning:** Size your reserve fund |

**Key insight:** CVaR is always *below* the floor — by definition, the average of the worst-case scenarios is worse than the boundary.

### When to care
- **High uncertainty feature?** CVaR tells you how deep the disaster can go
- **Multiple features with similar risk?** CVaR helps you stack contingencies
- **Board discussion?** "We budget with the business value floor, but prepare for CVaR in our downside scenario"


In [ ]:
# ── CVaR: Worst-Case Average ────────────────────────────────────────
show.downside_risk(result)

---

## 5. Sensitivity Test: How Uncertainty Changes Your Floor

The uncertainty parameter is *critical*—it controls the spread of outcomes. When you increase uncertainty, your business value floor drops. Here's what that looks like:


In [ ]:
# ── Uncertainty Impact Comparison ────────────────────────────────────
plot_uncertainty_comparison(checkout_feature);

**How to interpret the comparison**

Below you see three versions of the same feature with different uncertainty assumptions:

**Left panel (Low uncertainty = 10%):** Narrow histogram. Most outcomes cluster near the expected value. **Your business value floor is high.** You can make confident plans.

**Middle panel (Medium uncertainty = 30%):** Wider histogram. More spread. **Your business value floor drops.** You need more contingency budget.

**Right panel (High uncertainty = 50%):** Very wide histogram. The tail is far from the mean. **Your business value floor drops significantly.** Only now do you see the real downside risk.

**Key lesson:** The width of the histogram (and how far the business value floor is from Expected Value) tells you whether your assumptions are solid or shaky.

- **Range badge:** Shows P95 − P5 (the middle 90% of outcomes)
- **Red line:** Business Value Floor — notice how it shifts left as uncertainty increases
- **Expected value:** Stays constant because we haven't changed the base assumption, only confidence in it

> **Your next question:** "Should I increase uncertainty?"
> - Yes, if: You've had recent market shifts, you're targeting a new user segment, or technical risk is high
> - No, if: You have strong historical data and the feature is similar to past releases

> [T01 Distribution Guide](tutorial/01-distribution-guide.ipynb) digs deeper into *why* uncertainty matters and *how* to choose it.


| # | Notebook | What you learn |
|:-:|----------|----------------|
| 01 | **Getting Started** | ← You are here: one feature, one decision in 10 minutes |
| T01 | [Distribution Guide](tutorial/01-distribution-guide.ipynb) | Deep dive: Normal vs. Lognormal vs. Beta, floor/CVaR impact, Beta box plot |
| 02 | [Blockchain Case Study](02-blockchain-case-study.ipynb) | Three competing hypotheses, full EUR business case |
| A01 | [Portfolio Advisor](advanced/01-portfolio-advisor.ipynb) | Feature ranking, solver comparison (ILP/Exact/Greedy), runtime scaling, HHI |
| A02 | [Portfolio Risk Dashboard](advanced/02-portfolio-risk-dashboard.ipynb) | Risk layers L1/L2/L3, LLP impact, stress scenarios, budget risk path |


---

## Limits of This Simulation

Every model is a simplification. Before you act on these numbers, keep three things in mind:

- **Outputs depend on input assumptions.** If your expected user count or conversion rate is wrong, the simulation results are wrong too. Monte Carlo does not fix bad inputs — it shows you the *range* of outcomes given those inputs.
- **Distributions are approximations.** The model assumes user behaviour follows a statistical pattern (Normal or Lognormal). In reality, markets can shift in ways no distribution captures — a competitor launch, a regulatory change, or a viral moment.
- **Correlation is not causation.** When features share risk factors (market shock, platform dependency), the simulation models this as correlation. But real-world dependencies can be more complex than any correlation matrix.

> **Use these results as decision support, not as predictions.** The value of simulation is not that it gives you the "right" answer — it is that it shows you *how wrong* a single-point estimate can be.

---

## Summary

- You have modeled a feature with 3 inputs.
- You have simulated 10,000 scenarios.
- You can use Expected Value, Business Value Floor, and P95 for decision-making.

### Recommended next step
1. [T01 Distribution Guide](tutorial/01-distribution-guide.ipynb)
3. [02 Blockchain Case Study](02-blockchain-case-study.ipynb)


---

## Glossary

This notebook keeps a short working glossary. For the full reference, open [GLOSSARY.ipynb](GLOSSARY.ipynb).

---
**Next:** [NB 02: Blockchain Case Study](02-blockchain-case-study.ipynb)
